In [9]:
import pandas as pd
import numpy as np


df=pd.read_csv("../data/Interview_prediction.csv")
print("Dataset loaded successfully")

Dataset loaded successfully


In [10]:
df.shape

(1950, 10)

In [11]:
df.columns

Index(['Skill_Match_Percent', 'Preferred_Skill_Match_Percent',
       'Experience_Score', 'Project_Match_Percent', 'Education_Score',
       'ATS_Score', 'Experience_Years', 'Job_Role', 'Interview_Prediction',
       'Interview_Probability_Percent'],
      dtype='str')

In [12]:
df.isnull().sum()

Skill_Match_Percent              0
Preferred_Skill_Match_Percent    0
Experience_Score                 0
Project_Match_Percent            0
Education_Score                  0
ATS_Score                        0
Experience_Years                 0
Job_Role                         0
Interview_Prediction             0
Interview_Probability_Percent    0
dtype: int64

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df["Interview_Prediction"].value_counts()

Interview_Prediction
Likely      1154
Unlikely     796
Name: count, dtype: int64

In [15]:
df["Job_Role"].value_counts()

Job_Role
Database Administrator       150
DevOps Engineer              150
Machine Learning Engineer    150
Cloud Engineer               150
Python Developer             150
Data Scientist               150
QA Engineer                  150
Frontend Developer           150
Full Stack Developer         150
Backend Developer            150
Data Analyst                 150
Cybersecurity Analyst        150
Mobile App Developer         150
Name: count, dtype: int64

In [16]:
# Features
feature_columns = [
    "Skill_Match_Percent",
    "Preferred_Skill_Match_Percent",
    "Experience_Score",
    "Project_Match_Percent",
    "Education_Score",
    "ATS_Score",
    "Experience_Years",
    "Job_Role"
]

X = df[feature_columns]

# Target
y = df["Interview_Prediction"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1950, 8)
Target shape: (1950,)


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "Skill_Match_Percent",
    "Preferred_Skill_Match_Percent",
    "Experience_Score",
    "Project_Match_Percent",
    "Education_Score",
    "ATS_Score",
    "Experience_Years"
]

categorical_features = ["Job_Role"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (1560, 8)
Testing: (390, 8)


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

model4 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

In [21]:
model4.fit(X_train, y_train)

print("Model 4 training completed successfully!")

Model 4 training completed successfully!


In [22]:
y_pred = model4.predict(X_test)

print("Predictions generated successfully!")
print(y_pred[:10])

Predictions generated successfully!
['Unlikely' 'Unlikely' 'Unlikely' 'Likely' 'Unlikely' 'Likely' 'Unlikely'
 'Unlikely' 'Unlikely' 'Likely']


In [23]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)

print("Model 4 Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model 4 Accuracy: 81.03 %

Classification Report:
              precision    recall  f1-score   support

      Likely       0.86      0.82      0.84       231
    Unlikely       0.75      0.80      0.77       159

    accuracy                           0.81       390
   macro avg       0.80      0.81      0.81       390
weighted avg       0.81      0.81      0.81       390


Confusion Matrix:
[[189  42]
 [ 32 127]]


In [24]:
y_probability = model4.predict_proba(X_test)

print(y_probability[:5])

[[0.215 0.785]
 [0.41  0.59 ]
 [0.08  0.92 ]
 [0.905 0.095]
 [0.47  0.53 ]]


In [25]:
print(model4.named_steps["classifier"].classes_)

['Likely' 'Unlikely']


In [26]:
likely_index = list(
    model4.named_steps["classifier"].classes_
).index("Likely")

interview_probability = y_probability[:, likely_index] * 100

In [27]:
new_candidate = pd.DataFrame({
    "Skill_Match_Percent": [89.17],
    "Preferred_Skill_Match_Percent": [100],
    "Experience_Score": [100],
    "Project_Match_Percent": [90],
    "Education_Score": [100],
    "ATS_Score": [88],
    "Experience_Years": [4],
    "Job_Role": ["Data Scientist"]
})

prediction = model4.predict(new_candidate)[0]

probabilities = model4.predict_proba(new_candidate)[0]

likely_index = list(
    model4.named_steps["classifier"].classes_
).index("Likely")

probability = probabilities[likely_index] * 100

print("========================================")
print("INTERVIEW PREDICTION")
print("========================================")
print("Prediction:", prediction)
print("Interview Probability:", round(probability, 2), "%")

INTERVIEW PREDICTION
Prediction: Likely
Interview Probability: 95.5 %


In [28]:
y_probability = model4.predict_proba(X_test)

print("First 5 probability predictions:")
print(y_probability[:5])

print("\nModel classes:")
print(model4.named_steps["classifier"].classes_)

First 5 probability predictions:
[[0.215 0.785]
 [0.41  0.59 ]
 [0.08  0.92 ]
 [0.905 0.095]
 [0.47  0.53 ]]

Model classes:
['Likely' 'Unlikely']


In [29]:
likely_index = list(
    model4.named_steps["classifier"].classes_
).index("Likely")

interview_probability = y_probability[:, likely_index] * 100

print("\nFirst 10 Interview Probabilities:")
print(interview_probability[:10])


First 10 Interview Probabilities:
[21.5 41.   8.  90.5 47.  96.  18.  27.5  6.5 53. ]


In [30]:
import joblib

joblib.dump(model4, "interview_prediction.pkl")

print("Model 4 saved successfully.")

Model 4 saved successfully.
